In [ ]:
import json
import math
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from tqdm.auto import tqdm

from scipy.stats import linregress, pearsonr, t
import scipy.stats as stats

import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.anova as anova

In [ ]:
# Model and agent names
AGENT_TYPES = {
    1: 'low_danger_high_difficulty',    # low danger, high difficulty
    2: 'high_danger_low_difficulty',    # high danger, low difficulty
    3: 'low_danger_low_difficulty',     # low danger, low difficulty
    4: 'high_danger_high_difficulty'    # high danger, high difficulty
}

MODEL_MAPPING = {
    'A': 'mr_2_1_10',
    'B': 'mr_5_2_20', 
    'C': 'unlimited'
}

# 1. Data Processing

## 1.1. Mean ratings

In [ ]:
#  Import paerticipant ratings
df_ratings = pd.read_csv('../data/df_ratings.csv')

print(f"Total number of entries: {len(df_ratings)}")

In [ ]:
# mean_ratings per video (trajectory_unit)

mean_ratings = (
    df_ratings
    .groupby('trajectory_unit', as_index=False)
    .agg(
        trajectory_model=('trajectory_model', 'first'),
        trajectory_unit=('trajectory_unit', 'first'),
        trajectory = ('trajectory', 'first'),
        video_url = ('video_url', 'first'),
        agent=('agent', 'first'),
        amplitude_condition=('amplitude_condition', 'first'),
        danger_condition=('danger_condition', 'first'),
        map_difficulty_condition=('map_difficulty_condition', 'first'),
        rating_danger=('rating_danger', 'mean'),
        rating_danger_std=('rating_danger', 'std'),
        rating_map_difficulty=('rating_map_difficulty', 'mean'),
        rating_map_difficulty_std=('rating_map_difficulty', 'std'),
        rating_enjoyment=('rating_enjoyment', 'mean'),
        rating_enjoyment_median=('rating_enjoyment', 'median'),
        rating_enjoyment_std=('rating_enjoyment', 'std'),
        variance_rating_enjoyment=('rating_enjoyment', 'var'),
        danger_score=('danger_score', 'mean'),
        map_difficulty_score=('map_difficulty_score', 'mean'),
        count=('trajectory_unit', 'count')
    )
)

mean_ratings['trajectory_danger_condition'] = (
    mean_ratings['trajectory_model'] + '_' +
    mean_ratings['amplitude_condition'] + '_' +
    mean_ratings['danger_condition']
)

mean_ratings['trajectory_map_difficulty_condition'] = (
    mean_ratings['trajectory_model'] + '_' +
    mean_ratings['amplitude_condition'] + '_' +
    mean_ratings['map_difficulty_condition']
)

mean_ratings = mean_ratings[mean_ratings['count'] > 10]
print(f"Number of unique map+trajectories: {len(mean_ratings)}")
mean_ratings.describe()


## 1.2. Pairwise comparison

In [ ]:
# Import pairwise comparisons

pairwise_enjoyment_df = pd.read_csv('../data/df_pairwise_enjoyment.csv')
pairwise_danger_df = pd.read_csv('../data/df_pairwise_danger.csv')
pairwise_map_difficulty_df = pd.read_csv('../data/df_pairwise_map_difficulty.csv')

In [ ]:
# Compute and visualize pairwise win matrices for agent comparisons across enjoyment, danger, and map difficulty.

def compute_win_matrix(pairwise_df):
    all_agents = sorted(set(pairwise_df['agent_1'].unique().tolist() + pairwise_df['agent_2'].unique().tolist()))
    agent_names = [AGENT_TYPES[a] for a in all_agents]

    win_matrix = pd.DataFrame(
        np.zeros((len(all_agents), len(all_agents))),
        index=agent_names,
        columns=agent_names
    )

    for i, agent_i in enumerate(all_agents):
        for j, agent_j in enumerate(all_agents):
            if agent_i == agent_j:
                win_matrix.iloc[i, j] = np.nan
            else:
                wins_i_1 = pairwise_df[
                    (pairwise_df['agent_1'] == agent_i) & 
                    (pairwise_df['agent_2'] == agent_j) &
                    (pairwise_df['response'] == 0)
                ].shape[0]
                wins_i_2 = pairwise_df[
                    (pairwise_df['agent_2'] == agent_i) & 
                    (pairwise_df['agent_1'] == agent_j) &
                    (pairwise_df['response'] == 1)
                ].shape[0]
                total_comparisons = pairwise_df[
                    ((pairwise_df['agent_1'] == agent_i) & (pairwise_df['agent_2'] == agent_j)) |
                    ((pairwise_df['agent_2'] == agent_i) & (pairwise_df['agent_1'] == agent_j))
                ].shape[0]
                if total_comparisons > 0:
                    win_matrix.iloc[i, j] = (wins_i_1 + wins_i_2) / total_comparisons * 100
                else:
                    win_matrix.iloc[i, j] = np.nan
    return win_matrix

# Generate matrices
win_matrix_enjoyment = compute_win_matrix(pairwise_enjoyment_df)
win_matrix_danger = compute_win_matrix(pairwise_danger_df)
win_matrix_map_difficulty = compute_win_matrix(pairwise_map_difficulty_df)

# Set up side-by-side square heatmaps, slightly smaller but bigger numbers, new colormap
labels = ["Enjoyment", "Danger", "Map Difficulty"]
win_matrices = [win_matrix_enjoyment, win_matrix_danger, win_matrix_map_difficulty]

n = len(win_matrices)
n_agents = win_matrix_enjoyment.shape[0]
fig, axes = plt.subplots(1, n, figsize=(n * 5.5, 5.5), constrained_layout=True)  # smaller plots

if n == 1:
    axes = [axes]

for i in range(n):
    sns.heatmap(
        win_matrices[i],
        annot=True,
        fmt='.1f',
        cmap="coolwarm_r",                        # red to blue colormap
        vmin=0, vmax=100,
        linewidths=0.4,                           # keep lines fine
        square=True,
        ax=axes[i],
        annot_kws={"size": 10},                   # bigger annotation font
        cbar=(i == n-1),
        cbar_kws={'label': '% times agent in row beats agent in column'} if i == n-1 else None
    )
    axes[i].set_title(f"{labels[i]} Win Matrix")

plt.show()


# 2. Inter-rater reliability

In [ ]:
# Split-half reliability estimation using permutation and Spearman–Brown correction

def spearman_brown(r_half):
    return (2 * r_half) / (1 + r_half) if r_half < 1 else 1

def split_half_reliability(data, stimulus_col, participant_col, rating_col, 
                           n_iter=1000, random_state=42, min_raters=4, verbose=True):
    """
    Estimate split-half (Spearman–Brown corrected) reliability by randomly splitting raters.

    Parameters
    ----------
    data : pd.DataFrame
        Long-format rating data.
    stimulus_col, participant_col : str
        Column names for stimulus and participant.
    rating_col : str
        Column name for rating D.V.
    n_iter : int
        Number of permutations.
    min_raters : int
        Minimum number of raters per stimulus to include in the analysis.
    Returns
    -------
    dict with results and r_SB distribution
    """
    rng = np.random.default_rng(random_state)
    # Only use stimuli with at least min_raters
    by_stim = data.groupby(stimulus_col)
    n_raters_by_stim = by_stim[participant_col].nunique()
    good_stimuli = n_raters_by_stim[n_raters_by_stim >= min_raters].index
    data = data[data[stimulus_col].isin(good_stimuli)].copy()
    stim_list = sorted(data[stimulus_col].unique())

    r_sb_list = []
    miss_count = 0

    for i in tqdm(range(n_iter), desc=f'Split-half {rating_col}', disable=not verbose):
        # For each stimulus, randomly split raters in half; if odd, split as evenly as possible
        means_A = []
        means_B = []
        valid_stims = []
        for stim in stim_list:
            stim_dat = data.loc[data[stimulus_col] == stim, [participant_col, rating_col]]
            # Remove any duplicated ratings (should not happen, but safe)
            stim_dat = stim_dat.drop_duplicates(subset=[participant_col])
            raters = stim_dat[participant_col].values
            n = len(raters)
            if n < min_raters:
                continue  # skip, in case made smaller via filtering
            idx = rng.permutation(n)
            split = n // 2
            A = stim_dat.iloc[idx[:split]][rating_col].values
            B = stim_dat.iloc[idx[split:]][rating_col].values
            if len(A) == 0 or len(B) == 0:
                # This split isn't valid, skip
                continue
            means_A.append(np.mean(A))
            means_B.append(np.mean(B))
            valid_stims.append(stim)
        # Correlate across stimuli for this split
        if len(means_A) < 2:
            miss_count += 1
            continue  # skip, can't compute correlation on <2 stimuli
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            r_half = np.corrcoef(means_A, means_B)[0,1]
        if not np.isfinite(r_half):
            miss_count += 1
            continue
        r_sb = spearman_brown(r_half)
        r_sb_list.append(r_sb)
    r_sb_arr = np.array(r_sb_list)
    summary = {
        'r_SB_median': np.median(r_sb_arr),
        'r_SB_mean': np.mean(r_sb_arr),
        'r_SB_95ci_lower': np.percentile(r_sb_arr, 2.5),
        'r_SB_95ci_upper': np.percentile(r_sb_arr, 97.5),
        'n_permutations': len(r_sb_arr),
        'n_invalid': miss_count
    }
    return summary, r_sb_arr

# Prepare long-format dataframe
long_df = df_ratings.copy()
long_df = long_df.rename(columns={'participant_id': 'participant', 'trajectory_unit': 'stimulus'})

results_reliability = {}

for dv in ['rating_enjoyment', 'rating_danger', 'rating_map_difficulty']:
    print(f"================================================\nSplit-half reliability (Spearman-Brown) for {dv}\n================================================")
    data = long_df.dropna(subset=[dv])
    n_by_stim = data.groupby('stimulus')[dv].count()
    n_bar = n_by_stim.mean()
    n_min, n_max = int(n_by_stim.min()), int(n_by_stim.max())
    print(f"  Avg raters per stimulus n̄={n_bar:.2f} (min={n_min}, max={n_max})")
    res, r_sb_dist = split_half_reliability(
        data,
        stimulus_col='stimulus',
        participant_col='participant',
        rating_col=dv,
        n_iter=10000,
        random_state=42,
        min_raters=4,   # You may adjust for your data
        verbose=False
    )
    print(f"  Estimated full-sample reliability (median Spearman–Brown r): {res['r_SB_median']:.2f}")
    print(f"  95% CI: [{res['r_SB_95ci_lower']:.2f}, {res['r_SB_95ci_upper']:.2f}]   (N permutations: {res['n_permutations']})\n")
    results_reliability[dv] = res


# 3. Visualization

## 3.1. Distribution of enjoyment ratings

In [ ]:
# [PAPER FIGURE 2] Distribution of enjoyment ratings by trajectory: raw dots (gradient, sorted by mean) plus mean+95CI per video (colored gradient)

# Prepare data: only keep rows with both ratings and units
df_plot = df_ratings[df_ratings['rating_enjoyment'].notna() & df_ratings['trajectory_unit'].notna()]

# Compute mean for each unit (video) and sort
means = df_plot.groupby('trajectory_unit')['rating_enjoyment'].mean()
units_sorted = means.sort_values(ascending=False).index.tolist()

# Make plot dot/color gradient
cmap = LinearSegmentedColormap.from_list(
    "custom_teal_to_darkteal",
    ['#316a6c', '#42a3b9'],
    N=len(units_sorted) + 6
)
gradient_colors = [mcolors.to_hex(cmap(i + 6)) for i in range(len(units_sorted))]

dot_traces = []
mean_error_traces = []
DOT_X_OFFSET = 0.2

for i, unit in enumerate(units_sorted):
    vals = df_plot[df_plot['trajectory_unit'] == unit]['rating_enjoyment']
    x_jitter = np.random.normal(loc=i + DOT_X_OFFSET, scale=0.09, size=len(vals))
    dot_traces.append(go.Scatter(
        x=x_jitter, y=vals.values,
        mode='markers',
        marker=dict(color=gradient_colors[i], size=6),
        opacity=0.5,
        showlegend=False,
        hoverinfo='skip'
    ))
    n = len(vals)
    mean = np.mean(vals)
    h = t.ppf(0.975, n-1) * (np.std(vals, ddof=1) / np.sqrt(n)) if n > 1 else 0
    mean_error_traces.append(go.Scatter(
        x=[i], y=[mean],
        mode='markers',
        marker=dict(color=gradient_colors[i], size=12),
        error_y=dict(
            type='data', array=[h], visible=True, color=gradient_colors[i], thickness=4, width=8
        ),
        name=f'Video {i + 1}<br>({unit})',  # Directly use the trajectory_unit string
        opacity=0.95,
        showlegend=False
    ))

fig = go.Figure(data=dot_traces + mean_error_traces)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=list(range(len(units_sorted))),
        ticktext=[str(i+1) for i in range(len(units_sorted))],
        tickfont=dict(size=14),
        showline=True, showgrid=True, zeroline=False,
        mirror=False, linewidth=0.8, linecolor='black',
        ticks='outside', ticklen=4,
    ),
    yaxis=dict(
        title=dict(text="Enjoyment ratings", font=dict(size=16)),
        showline=True, showgrid=True, zeroline=False,
        mirror=False, linewidth=0.8, linecolor='black',
        ticks='outside', ticklen=4, tickfont=dict(size=14),
    ),
    width=1400, height=600,
    margin=dict(l=10, r=10, t=40, b=40),
    plot_bgcolor='white'
)

fig.update_xaxes(range=[-0.8, len(units_sorted)])
fig.show()

In [ ]:
# Min and max of mean ratings
mean_ratings.describe()

In [ ]:
# Min and max of mean ratings
M = mean_ratings.groupby('trajectory').agg({'rating_map_difficulty': ['min', 'max']})
M['rating_map_difficulty_diff'] = M['rating_map_difficulty']['max'] - M['rating_map_difficulty']['min']
M.describe()

## 3.2. Correlations

In [ ]:
# Correlations between ratings and scores, with relabeled axes/titles

corr_cols = [
    "rating_danger",
    "danger_score",
    "rating_map_difficulty",
    "map_difficulty_score",
    "rating_enjoyment",
    "rating_enjoyment_std"
]
corr_data = mean_ratings[corr_cols]

# Set up pretty names for columns
pretty_labels = [
    "Danger rating",
    "Computed danger score",
    "Map difficulty rating",
    "Comuted map difficulty",
    "Enjoyment rating",
    "Standard deviation of enjoyment rating"
]
pretty_label_dict = dict(zip(corr_cols, pretty_labels))
corr_data_pretty = corr_data.copy()
corr_data_pretty.columns = pretty_labels

# Compute correlation matrix and p-values
corr_matrix = corr_data.corr()
pvals = np.ones_like(corr_matrix, dtype=float)

for i, col1 in enumerate(corr_cols):
    for j, col2 in enumerate(corr_cols):
        if i == j:
            pvals[i, j] = 0
        elif i < j:
            mask = corr_data[[col1, col2]].dropna()
            if len(mask) > 2:
                r, p = pearsonr(mask[col1], mask[col2])
                pvals[i, j] = p
                pvals[j, i] = p
            else:
                pvals[i, j] = np.nan
                pvals[j, i] = np.nan

# Prepare annotation matrix: correlation + significance stars, use pretty_labels order
def significance_stars(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    elif p < 0.9:
        return f"\np={p:.2f}"
    else:
        return ""

annot = corr_matrix.round(2).astype(str)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        stars = significance_stars(pvals[i, j])
        if i == j:
            annot.iloc[i, j] = ""
        else:
            annot.iloc[i, j] += stars
annot_pretty = annot.copy()
annot_pretty.index = pretty_labels
annot_pretty.columns = pretty_labels

corr_matrix_pretty = corr_matrix.copy()
corr_matrix_pretty.index = pretty_labels    
corr_matrix_pretty.columns = pretty_labels

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix_pretty, 
    annot=annot_pretty, 
    fmt="", 
    cmap="coolwarm", 
    vmin=-1, vmax=1,
    cbar_kws={"label": "Pearson r"}
)
plt.title("Correlation Matrix of Ratings and Computed Scores\n(* p<0.05 ** p<0.01 *** p<0.001)", fontsize=15)
plt.yticks(rotation=0)
plt.xticks(rotation=45, ha='right')  # Make X axis labels at 45 degrees
plt.show()

# Pairwise scatterplots with relabeled columns
g = sns.pairplot(corr_data_pretty, kind="reg", diag_kind="kde")
for ax in g.axes[-1, :]:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.suptitle("Pairwise Correlations between Danger, Score, Map Difficulty, and Enjoyment", y=1.03)
plt.show()

In [ ]:
# Print the 95% CI for correlations among rating_enjoyment, rating_map_difficulty, and rating_danger

pairs = [
    ('rating_enjoyment', 'rating_map_difficulty'),
    ('rating_enjoyment', 'rating_danger'),
    ('rating_map_difficulty', 'rating_danger'),
    ('rating_map_difficulty', 'danger_score'),
    ('map_difficulty_score', 'danger_score')
]

df_CI = mean_ratings[['rating_enjoyment', 'rating_map_difficulty', 'rating_danger', 'danger_score', 'map_difficulty_score']].dropna()
n = len(df_CI)

if n > 3:
    for col1, col2 in pairs:
        r, p = pearsonr(df_CI[col1], df_CI[col2])

        # Fisher r-to-z CI
        z = np.arctanh(r)
        se = 1 / math.sqrt(n - 3)
        z_crit = 1.96  # 95%
        lo_z, hi_z = z - z_crit * se, z + z_crit * se
        lo, hi = np.tanh([lo_z, hi_z])

        print(f"Pearson r({col1} vs {col2}) = {r:.3f}, 95% CI [{lo:.3f}, {hi:.3f}], p = {p:.4f}, n = {n}")
else:
    print("Not enough paired data to compute Fisher CI for all pairs (need n > 3).")

## 3.3. Plot ratings vs score

In [ ]:
# Plotting functions

N_RATINGS = 48

CI_COLUMNS = {
    'rating_danger': 'rating_danger_std',
    'rating_map_difficulty': 'rating_map_difficulty_std',
    'rating_enjoyment': 'rating_enjoyment_std',
}

for rating_col, std_col in CI_COLUMNS.items():
    mean_ratings[f'{rating_col}_95ci'] = mean_ratings[std_col] / np.sqrt(N_RATINGS)

COLORS = {
    'danger': '#FF7BAC',
    'danger_fill': 'rgba(255,105,180,0.16)',
    'mapdiff': '#FFBA00',
    'mapdiff_fill': 'rgba(255,186,0,0.18)',
    'grey': '#888888',
    'grey_fill': 'rgba(136,136,136,0.1)',
    'blue': '#53A1B7',
}

AXIS_STYLE = {
    'gridcolor': 'rgba(211,211,211,0.3)',
    'tickfont': dict(size=14),
    'title_font': dict(size=16),
}


def rating_ci(column):
    """Return the precomputed 95% CI series for a rating column."""
    return mean_ratings.get(f'{column}_95ci')


def fit_regression_with_ci(x, y, n_points=100):
    """Fit a line and compute its 95% confidence band."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), n_points)
    y_line = slope * x_line + intercept

    n = len(x)
    dof = n - 2
    residuals = y - (slope * x + intercept)
    resid_std_err = np.sqrt(np.sum(residuals ** 2) / dof)
    mean_x = np.mean(x)
    se_y_line = resid_std_err * np.sqrt(
        1 / n + (x_line - mean_x) ** 2 / np.sum((x - mean_x) ** 2)
    )
    ci_delta = stats.t.ppf(0.975, dof) * se_y_line

    return {
        'x_line': x_line,
        'y_line': y_line,
        'ci_upper': y_line + ci_delta,
        'ci_lower': y_line - ci_delta,
        'r2': r_value ** 2,
        'p_value': p_value,
        'std_err': std_err,
    }


def error_bar(axis_error, color=None, thickness=None, width=None):
    if axis_error is None:
        return None

    error = dict(type='data', array=axis_error, visible=True)
    if color is not None:
        error['color'] = color
    if thickness is not None:
        error['thickness'] = thickness
    if width is not None:
        error['width'] = width
    return error


def add_regression_panel(fig, *, col, x, y, x_label, y_label, x_title, y_title,
                         marker_color, line_color, fill_color, trace_name,
                         line_name='Linear Fit', x_error=None, y_error=None,
                         x_error_color=None, y_error_color=None,
                         error_thickness=None, error_width=None):
    fit = fit_regression_with_ci(x, y)
    customdata = np.stack([y_error], axis=-1) if y_error is not None else None
    hovertemplate = (
        f'<b>{x_label}</b>: %{{x:.2f}}<br>'
        f'<b>{y_label}</b>: %{{y:.2f}}'
        + ("<br><b>95% CI</b>: %{customdata[0]:.2f}" if y_error is not None else '')
        + '<extra></extra>'
    )

    scatter_kwargs = dict(
        x=x,
        y=y,
        mode='markers',
        marker=dict(color=marker_color, size=10, line=dict(width=1, color='white')),
        name=trace_name,
        hovertemplate=hovertemplate,
        customdata=customdata,
        showlegend=False,
    )
    x_error_bar = error_bar(x_error, x_error_color, error_thickness, error_width)
    y_error_bar = error_bar(y_error, y_error_color, error_thickness, error_width)
    if x_error_bar is not None:
        scatter_kwargs['error_x'] = x_error_bar
    if y_error_bar is not None:
        scatter_kwargs['error_y'] = y_error_bar

    fig.add_trace(go.Scatter(**scatter_kwargs), row=1, col=col)
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([fit['x_line'], fit['x_line'][::-1]]),
            y=np.concatenate([fit['ci_upper'], fit['ci_lower'][::-1]]),
            fill='toself',
            fillcolor=fill_color,
            line=dict(color='rgba(0,0,0,0)'),
            hoverinfo='skip',
            showlegend=False,
            name='95% CI (fit)',
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=fit['x_line'],
            y=fit['y_line'],
            mode='lines',
            name=line_name,
            line=dict(color=line_color, width=3, dash='dot'),
            hoverinfo='skip',
            showlegend=False,
        ),
        row=1,
        col=col,
    )

    axis_suffix = '' if col == 1 else str(col)
    fig.add_annotation(
        xref=f'x{axis_suffix} domain',
        yref=f'y{axis_suffix} domain',
        x=0.05,
        y=0.99,
        showarrow=False,
        align='left',
        bgcolor='rgba(255,255,255,0.85)',
        text=f"<b>R² = {fit['r2']:.2f}</b>",
        font=dict(size=17),
        row=1,
        col=col,
    )

    fig.update_xaxes(title_text=x_title, **AXIS_STYLE, row=1, col=col)
    fig.update_yaxes(title_text=y_title, title_standoff=10, **AXIS_STYLE, row=1, col=col)
    return fit


def tickvals_10(min_val, max_val):
    start = max(0, 10 * int(np.floor(min_val / 10.0)))
    end = min(100, 10 * int(np.ceil(max_val / 10.0)))
    return list(np.arange(start, end + 1, 10))


def model_r2s(data, formulas):
    return [smf.ols(formula, data=data).fit().rsquared for formula in formulas.values()]

In [ ]:
# [PAPER FIGURE 3] Danger ratings vs computed danger estimates AND Map difficulty ratings vs computed map difficulty scores AND Map difficulty ratings vs Danger ratings (side-by-side with error bars for 95% CI)
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=("", "", ""),
    horizontal_spacing=0.05,
)

figure_3_panels = [
    dict(
        col=1,
        x=mean_ratings['danger_score'],
        y=mean_ratings['rating_danger'],
        y_error=rating_ci('rating_danger'),
        x_label='Computed Danger',
        y_label='Danger Rating',
        x_title='Computed danger estimates',
        y_title='Danger ratings',
        marker_color=COLORS['danger'],
        line_color=COLORS['danger'],
        fill_color=COLORS['danger_fill'],
        trace_name='Danger: Mean ± 95% CI',
        line_name='Danger Linear Fit',
    ),
    dict(
        col=2,
        x=mean_ratings['map_difficulty_score'],
        y=mean_ratings['rating_map_difficulty'],
        y_error=rating_ci('rating_map_difficulty'),
        x_label='Computed Map Difficulty',
        y_label='Map Difficulty Rating',
        x_title='Computed map difficulty estimates',
        y_title='Map difficulty ratings',
        marker_color=COLORS['mapdiff'],
        line_color=COLORS['mapdiff'],
        fill_color=COLORS['mapdiff_fill'],
        trace_name='Difficulty: Mean ± 95% CI',
        line_name='Difficulty Linear Fit',
    ),
    dict(
        col=3,
        x=mean_ratings['rating_danger'],
        y=mean_ratings['rating_map_difficulty'],
        x_error=rating_ci('rating_danger'),
        y_error=rating_ci('rating_map_difficulty'),
        x_error_color=COLORS['danger'],
        y_error_color=COLORS['mapdiff'],
        x_label='Danger Rating',
        y_label='Map Difficulty Rating',
        x_title='Danger ratings',
        y_title='Map difficulty ratings',
        marker_color=COLORS['grey'],
        line_color=COLORS['grey'],
        fill_color=COLORS['grey_fill'],
        trace_name='Map Difficulty vs Danger: Mean ± 95% CI',
        line_name='Difficulty~Danger Linear Fit',
    ),
]

for panel in figure_3_panels:
    add_regression_panel(fig, **panel)
    fig.update_yaxes(
        tickvals=tickvals_10(panel['y'].min(), panel['y'].max()),
        row=1,
        col=panel['col'],
    )

fig.update_layout(
    template='simple_white',
    font=dict(size=14),
    margin=dict(l=10, r=10, t=50, b=80),
    width=1650,
    height=500,
    title_text=None,
    showlegend=False,
    legend=dict(borderwidth=0),
)

fig.show()

In [ ]:
# [PAPER FIGURE 4] Enjoyment ratings vs danger ratings AND enjoyment ratings vs difficulty ratings (side-by-side with error bars for 95% CI)
# Also includes the model R2 bar/histogram as a third subplot.

NOISE_CEILING_ENJOYMENT = 0.568
NOISE_CEILING_ENJOYMENT_LOWER = 0.25
NOISE_CEILING_ENJOYMENT_UPPER = 0.78

models = {
    'Danger<br>Rating': 'rating_enjoyment ~ rating_danger',
    'Difficulty<br>Rating': 'rating_enjoyment ~ rating_map_difficulty',
    'Difficulty Rating<br>+ Danger Rating': 'rating_enjoyment ~ rating_map_difficulty + rating_danger',
    'Difficulty Rating<br>× Danger Rating': 'rating_enjoyment ~ rating_map_difficulty * rating_danger',
}
plotly_labels = list(models.keys())
r2s = model_r2s(mean_ratings, models)

fig = make_subplots(
    rows=1,
    cols=3,
    column_widths=[0.31, 0.31, 0.38],
    horizontal_spacing=0.05,
)

scatter_panels = [
    dict(
        col=1,
        x=mean_ratings['rating_danger'],
        y=mean_ratings['rating_enjoyment'],
        x_error=rating_ci('rating_danger'),
        y_error=rating_ci('rating_enjoyment'),
        x_error_color=COLORS['danger'],
        y_error_color=COLORS['blue'],
        error_thickness=2,
        error_width=5,
        x_label='Danger Rating',
        y_label='Enjoyment Rating',
        x_title='Danger ratings',
        y_title='Enjoyment ratings',
        marker_color=COLORS['grey'],
        line_color=COLORS['grey'],
        fill_color=COLORS['grey_fill'],
        trace_name='Enjoyment vs Danger: Mean ± 95% CI',
    ),
    dict(
        col=2,
        x=mean_ratings['rating_map_difficulty'],
        y=mean_ratings['rating_enjoyment'],
        x_error=rating_ci('rating_map_difficulty'),
        y_error=rating_ci('rating_enjoyment'),
        x_error_color=COLORS['mapdiff'],
        y_error_color=COLORS['blue'],
        error_thickness=2,
        error_width=5,
        x_label='Difficulty Rating',
        y_label='Enjoyment Rating',
        x_title='Map difficulty ratings',
        y_title='Enjoyment ratings',
        marker_color=COLORS['grey'],
        line_color=COLORS['grey'],
        fill_color=COLORS['grey_fill'],
        trace_name='Enjoyment vs Difficulty: Mean ± 95% CI',
    ),
]

for panel in scatter_panels:
    add_regression_panel(fig, **panel)

fig.add_trace(
    go.Bar(
        x=plotly_labels,
        y=r2s,
        marker_color=COLORS['blue'],
        width=0.6,
        showlegend=False,
    ),
    row=1,
    col=3,
)

fig.add_shape(
    type='rect',
    xref='x3 domain',
    yref='y3',
    x0=0,
    x1=1,
    y0=NOISE_CEILING_ENJOYMENT_LOWER,
    y1=NOISE_CEILING_ENJOYMENT_UPPER,
    fillcolor='rgba(100,100,100,0.18)',
    layer='below',
    line_width=0,
    row=1,
    col=3,
)
fig.add_shape(
    type='line',
    xref='x3',
    yref='y3',
    x0=-1,
    x1=3.5,
    y0=NOISE_CEILING_ENJOYMENT,
    y1=NOISE_CEILING_ENJOYMENT,
    line=dict(color='rgba(0,0,0,1)', dash='dash', width=3),
    layer='above',
    row=1,
    col=3,
)
fig.add_annotation(
    x=0.5,
    y=NOISE_CEILING_ENJOYMENT + 0.04,
    text='Noise ceiling R² = 0.57',
    xref='x3',
    yref='y3',
    showarrow=False,
    yshift=12,
    font=dict(size=15, color='grey'),
    align='left',
    bgcolor='rgba(255,255,255,0.0)',
    row=1,
    col=3,
)

fig.update_xaxes(
    title_font=dict(size=15),
    tickfont=dict(size=13),
    tickangle=0,
    tickvals=plotly_labels,
    ticktext=plotly_labels,
    range=[-0.5, len(plotly_labels) - 0.5],
    row=1,
    col=3,
)
fig.update_yaxes(
    range=[0, 0.8],
    title_text='Model R²',
    title_font=dict(size=15),
    gridcolor='rgba(211,211,211,0.3)',
    title_standoff=10,
    row=1,
    col=3,
)
fig.update_layout(
    template='simple_white',
    font=dict(size=14),
    margin=dict(l=10, r=10, t=50, b=80),
    width=1480,
    height=470,
    title_text=None,
    showlegend=False,
    legend=dict(borderwidth=0),
)

# fig.write_image(FIGURE_PATH+'enjoyment_corr_poster.pdf', width=1480, height=500)
fig.show()

# 4. Main correlations investigation


In [ ]:
# Function for mixed effect model to predict enjoyment from agent and damage level
def print_model_summary_and_r2(model):
    print('==================== Model summary ====================\n')
    print(model.summary())

    # Calculate variance components
    var_fixed = model.fittedvalues.var()
    var_random = float(model.cov_re.iloc[0, 0])  # Random effect variance
    var_resid = model.scale  # Residual variance

    # Marginal R² (fixed effects only)
    r2_marginal = var_fixed / (var_fixed + var_random + var_resid)

    # Conditional R² (fixed + random effects)
    r2_conditional = (var_fixed + var_random) / (var_fixed + var_random + var_resid)

    print(f"Marginal R² (fixed effects): {r2_marginal:.4f}")
    print(f"Conditional R² (fixed + random): {r2_conditional:.4f}")


In [ ]:
# Computed danger score explained additional variance in map difficulty ratings

model_1 = smf.ols(
    "rating_map_difficulty ~ map_difficulty_score",
    data=mean_ratings
).fit()

model_2 = smf.ols(
    "rating_map_difficulty ~ map_difficulty_score + danger_score",
    data=mean_ratings
).fit()

r2_1 = model_1.rsquared
r2_2 = model_2.rsquared
delta_r2 = r2_2 - r2_1

print("R² model 1:", r2_1)
print("R² model 2:", r2_2)
print("ΔR²:", delta_r2)

anova = sm.stats.anova_lm(model_1, model_2)
print(anova)

print('==================== Model enjoyment summary ====================\n')
print(model_2.summary())

In [ ]:
# Map difficulty rating explains some variance in enjoyment ratings

model_1 = smf.ols(
    "rating_enjoyment ~ rating_map_difficulty",
    data=mean_ratings
).fit()

model_2 = smf.ols(
    "rating_enjoyment ~ rating_danger",
    data=mean_ratings
).fit()

model_3 = smf.ols(
    "rating_enjoyment ~ rating_danger + rating_map_difficulty",
    data=mean_ratings
).fit()

print('==================== Model enjoyment summary ====================\n')
print(model_1.summary())

print('==================== Model enjoyment summary ====================\n')
print(model_2.summary())

print('==================== Model enjoyment summary ====================\n')
print(model_3.summary())
